In [ ]:
from collections.abc import Iterable, Mapping
from typing import Any

def explore_paths(data: Any, prefix: str = "") -> None:
    if isinstance(data, Mapping):
        for key, value in data.items():
            path = f"{prefix}.{key}" if prefix else str(key)
            explore_paths(value, path)

    elif isinstance(data, list):
        for index, item in enumerate(data[:3]):
            path = f"{prefix}[{index}]"
            explore_paths(item, path)

        if len(data) > 3:
            print(f"{prefix}[...] = {len(data)} total items")

    else:
        print(prefix, "=", repr(data), "| type:", type(data).__name__)


def collect_paths(data: Any, prefix: str = "") -> list[tuple[str, Any]]:
    paths = []

    if isinstance(data, Mapping):
        for key, value in data.items():
            path = f"{prefix}.{key}" if prefix else str(key)
            paths.extend(collect_paths(value, path))

    elif isinstance(data, list):
        for index, item in enumerate(data[:3]):
            path = f"{prefix}[{index}]"
            paths.extend(collect_paths(item, path))

    else:
        paths.append((prefix, data))

    return paths


def inspect_schema(rows: Iterable[Mapping[str, Any]], limit: int = 10) -> dict[str, dict[str, Any]]:
    schema: dict[str, dict[str, Any]] = {}

    for row_number, row in enumerate(rows, start=1):
        if row_number > limit:
            break

        for path, value in collect_paths(row):
            if path not in schema:
                schema[path] = {
                    "types": set(),
                    "examples": []
                }

            schema[path]["types"].add(type(value).__name__)

            if len(schema[path]["examples"]) < 3:
                schema[path]["examples"].append(value)

    return schema